# Preprocess and upload dataset to HuggingFace (Phase 1)

In [24]:
import json
import pandas as pd
import numpy as np


from huggingface_hub import login, whoami
from datasets import Dataset, DatasetDict, Sequence, Image
from pathlib import Path, PurePosixPath

Log into HuggingFace via access token

In [3]:
HF_TOKEN="<HF_TOKEN>"
login(token=HF_TOKEN)

print(whoami())

{'type': 'user', 'id': '6aad11231339b5a189922a50', 'name': 'choisongjun', 'fullname': 'HT', 'isPro': False, 'avatarUrl': '/avatars/481c7e7c8d0b9c678adf6db8ab7a351d.svg', 'orgs': [], 'auth': {'type': 'access_token', 'accessToken': {'displayName': 'HuggingFace-CreateDataset', 'role': 'fineGrained', 'createdAt': '2026-09-29T07:21:42.016Z', 'fineGrained': {'canReadGatedRepos': True, 'global': ['discussion.write'], 'scoped': [{'entity': {'_id': '6aad11231339b5a189922a50', 'type': 'user', 'name': 'choisongjun'}, 'permissions': ['repo.content.read', 'repo.access.read', 'repo.write', 'discussion.write']}]}}}}


**Phase 1.1:  Combine all cases into one .csv file**

Locate custom dataset

In [4]:
def find_dataset_root(dirname: str = "extracted_dataset_1", search_root: Path = None) -> Path:
    root = search_root or Path.cwd()

    direct = root / dirname
    if direct.is_dir():
        return direct

    matches = [p for p in root.rglob(dirname) if p.is_dir()]
    if matches:
        return matches[0]

    parent = root
    searched = [str(root)]
    for _ in range(4):
        parent = parent.parent
        searched.append(str(parent))
        matches = [p for p in parent.rglob(dirname) if p.is_dir()]
        if matches:
            return matches[0]

    raise FileNotFoundError(f"Could not find '{dirname}'. Searched under: {searched}")

DATASET_ROOT = find_dataset_root()
print("Using:", DATASET_ROOT)

Using: /home/tokuden/VGU_WS26_ClinicalProject_BHTBH/BHTBH/extracted_dataset_1


Walk through each disease folder in **DATASET_ROOT**, and build one combined table with columns `case_id, case_text, disease`.

In [5]:
def build_combined_dataset(diseases: list[str], dataset_root: Path) -> pd.DataFrame:
    def process_image_metadata(image_metadata_path: Path):
        # image_metadata.json is JSON Lines (one dict per line), not a single JSON array
        case_id_to_images = {}
        with open(image_metadata_path, "r") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                entry = json.loads(line)
                case_id = entry["case_id"]

                # raw file_path looks like "medical_datasets/<disease>/images/PMC5/PMC58/xxx.webp"
                # drop whatever the leading root segment is named, keep everything after it
                relative_parts = PurePosixPath(entry["file_path"]).parts[1:]
                resolved_path = str(dataset_root.joinpath(*relative_parts))

                # for every case_id, get a list of relevant images
                case_id_to_images.setdefault(case_id, []).append(resolved_path)

        return case_id_to_images

    all_rows = []

    for disease in diseases:
        disease_path = dataset_root / disease

        disease_df = pd.read_csv(disease_path / "cases_with_image.csv", dtype=str)
        image_paths = process_image_metadata(disease_path / "image_metadata.json")


        rows = pd.DataFrame({
            "case_id": disease_df["case_id"],
            "case_text": disease_df["case_text"],
            "disease": disease,
        })

        rows["images"] = rows["case_id"].map(lambda cid: image_paths.get(cid, []))

        n_missing = (rows["images"].str.len() == 0).sum()
        if n_missing:
            print(f"[warning] {disease}: {n_missing}/{len(rows)} cases had no matching image_metadata.json entry")

        all_rows.append(rows)

    all_cases_df = pd.concat(all_rows, ignore_index=True)

    return all_cases_df

In [6]:
DISEASES = sorted([p.name for p in DATASET_ROOT.iterdir() if p.is_dir()])
all_cases_df = build_combined_dataset(DISEASES, DATASET_ROOT)
print(f"\nTotal combined: {len(all_cases_df)} rows across {all_cases_df['disease'].nunique()} diseases")


Total combined: 95 rows across 8 diseases


**Phase 1.2:  Load AI model and prepare for preprocessing**

In [1]:
import torch
import copy
import os
from langchain_core.prompts import PromptTemplate
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, DynamicCache

/home/tokuden/VGU_WS26_ClinicalProject_BHTBH/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Prompt used for this project

In [2]:
# Real variables: {case_text}, {disease}, {example_output_format}
INSTRUCTION_PROMPT = """You are an expert clinician–educator.

You will receive a raw case report. Summarize the case into a JSON object for a high-quality vector database. Also make your own diagnosis from the text and justify it with reasoning and a direct quote from the case text. Follow the rules below strictly, without exceptions.

RULES (Read Carefully—No Exceptions)

1. Source Fidelity
- Use only facts from the case text.
- Do NOT invent, embellish, or “smooth out” missing data.
- Paraphrase prose into concise statements where helpful, but never add new facts.

2. Work Order
Work in this order: Metadata Skimming → Case Presentation → Diagnostic Reasoning → Final Diagnosis.

3. JSON Keys
Use exactly these top-level keys: "metadata", "case_presentation", "final_diagnosis", "diagnostic_reasoning".

4. "metadata"
- Read the case text once without summarizing it.
- "patients": one object per patient, each with "age" and "gender", as stated in the case text ("not stated" if missing). Use an empty list if no patient is mentioned.
- "images": one object per image the case mentions, each with "image_id" and "image_path". An image is a .webp file wrapped in “<>”, which is itself wrapped in “()”. Always include "images", even if the case mentions none (use an empty list).
- "image_id": the string “Image x”, where x starts at 1 and increases by 1 for each new image, in order of appearance.
- "image_path": the path to the image exactly as written in the case text, including the .webp extension.

5. "case_presentation"
- Read the case text again and summarize it, presenting only facts. Do not add outside knowledge or interpretation.
- Replace each image path with its image_id (e.g., “In Image 1”, instead of “In (<PMC93840.webp>)”).
- Keep the order of the case report (e.g., physical exam and labs before imaging).
- Stay as close as possible to the writing style of the case report.
- Use a maximum of 200 words.

6. "final_diagnosis"
- A single disease/entity name (e.g., sarcoidosis).
- No adjectives, punctuation, or explanatory text.

7. "diagnostic_reasoning"
- "reasoning": one short sentence, at most 25 words, explaining how the case leads to final_diagnosis.
- "direct_quote": a quote copied exactly from the case text that supports that sentence.

9. Output Format
- Output one valid JSON object and nothing else (no code fences, no extra text).
- Follow the template below exactly. Replace each UPPERCASE placeholder with real content, and repeat the patient and image objects as many times as needed.
- Escape any double quotes inside strings.

OUTPUT_TEMPLATE_JSON (Copy exactly)

{"metadata": {
    "patients": [
      {"age": "PATIENT_AGE", "gender": "PATIENT_GENDER"}
    ],
    "images": [
      {"image_id": "IMAGE_ID", "image_path": "IMAGE_PATH"}
    ]
  },
  "case_presentation": "SUMMARY_OF_CASE_TEXT",
  "final_diagnosis": "DISEASE_NAME",
  "diagnostic_reasoning": {
    "reasoning": "SHORT_SENTENCE_FOR_REASONING",
    "direct_quote": "QUOTE_DIRECTLY_FROM_CASE_TEXT"
  },
}
"""

Load the model

In [3]:
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

try:
    from transformers import core_model_loading
    core_model_loading.GLOBAL_WORKERS = 1
except Exception:
    pass

MODEL_ID = "Qwen/Qwen3.5-4B"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map={"": 0},   # keep everything on the GPU
)
model.eval()
print(f"Loaded {MODEL_ID} in 4-bit | GPU memory used: {torch.cuda.memory_allocated() / 1024**3:.2f} GiB")

W1004 14:44:52.380000 3557 .venv/lib/python3.12/site-packages/torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1004 14:44:52.409000 3557 .venv/lib/python3.12/site-packages/torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
Loading weights: 100%|██████████| 426/426 [00:03<00:00, 106.58it/s]


Loaded Qwen/Qwen3.5-4B in 4-bit | GPU memory used: 2.90 GiB


In [13]:
case_report = """A 63-year-old Thai male farmer residing in central Thailand presented with prolonged fever and non-productive cough for 3 months. He also complained of intermittent chest tightness at the left hemithorax and losing 4 kg of his weight. He was admitted to the local hospital and diagnosed with bacterial pneumonia at that time. However, his fever did not subside following 14 days of intravenous meropenem. He was, therefore, transferred to our hospital for further investigations. He was a former smoker of 25 pack-years quitted 2 years ago before the presentation. He had end-stage renal disease requiring deceased donor kidney transplantation 4 years ago. After transplantation, he was on immunosuppressive agents, including mycophenolate mofetil and cyclosporine. Acyclovir was also prescribed for herpes infection prophylaxis.
He was referred to our hospital. Physical examination showed an asthenic patient with anicteric sclera. His vital signs revealed a body temperature of 38 c, blood pressure of 100/60 mm Hg, a pulse rate of 90/minute, and a respiratory rate of 17/minute. His oxygen saturation was 95% at ambient air. Respiratory examination revealed decreased breath sound at the left upper lung zone without crackles or egophony. Other physical examinations were unremarkable.
Complete blood count showed mild anemia with a hemoglobin level of 10.1 g/dL and leukocytosis with white blood cells count of 22 x 103/microL, 91% neutrophils. Blood chemistry tests remained within normal ranges. The chest radiograph (<PMC7455599_IDR-13-2957-g0001_A_1_2.webp>) and contrast-enhanced chest CT (<PMC7455599_IDR-13-2957-g0001_B_2_2.webp>) showed a large mass in the left upper lobe. A transthoracic core-needle biopsy of the left upper lobe mass was performed. The biopsy specimens were sent for pathological and microbiological studies. The histopathology revealed non-small cell carcinoma admixed with necrotic tissues. The immunohistochemical studies for cytokeratin (CK) AE1/AE3, CK7, and thyroid transcription factor-1 (TTF-1) were positive, compatible with primary lung adenocarcinoma. The Brown-Hopp stain revealed a single cluster of gram-negative bipolar rods with safety pin appearance characteristics of B. pseudomallei. (Figure 2A-D). B. pseudomallei subsequently grew on tissue culture. His serum antibody titer against B. pseudomallei was positive with a titer of 1:1280. His blood culture for aerobes and serum cryptococcal antigen were negative. The patient was finally diagnosed with stage IIIA non-small cell lung cancer (T4N0M0) co-existing with localized pulmonary melioidosis. The molecular analysis did not detect mutations of epidermal growth factor receptor (EGFR) genes and the rearrangement of anaplastic lymphoma kinase (ALK) gene.
After 3 weeks of intravenous ceftazidime, his fever subsided, and his well-being improved. He gradually gained his weight. Oral co-trimoxazole was sequentially prescribed for 20 weeks after the completion of intravenous ceftazidime treatment. He underwent surgery for left thoracotomy and left upper lobectomy. However, the procedure was not accomplished due to severe adhesion surrounding the entire left lung and great vessels. After surgery, he received concurrent chemoradiation therapy with a paclitaxel-based regimen for adenocarcinoma. Nevertheless, he developed progression of disease soon after. Three cycles of palliative chemotherapy using pemetrexed were delivered. Unfortunately, he finally passed away after 2 years after the initial presentation."""

Set up KV cache for **INSTRUCTION_PROMPT**

Relevant documentations:

Chat templates: https://huggingface.co/docs/transformers/main/en/chat_templating

KV Cache:   https://huggingface.co/docs/transformers/en/kv_cache

In [19]:
# Function to render query, can be used for caching (1st run) and querying (subsequent runs)
def render(INSTRUCTION_PROMPT, CASE_PROMPT, add_generation_prompt):

    MSGS = [{"role": "system", "content": INSTRUCTION_PROMPT}, {"role": "user", "content": CASE_PROMPT}]

    return tokenizer.apply_chat_template(
        MSGS,
        add_generation_prompt=add_generation_prompt,    # User tweaks behavior
        enable_thinking=False,          # No <think></think> in response.
        tokenize=False,                 # Don't turn into vector embeddings just yet, need to extract INSTRUCTION_PROMPT for caching first.
        return_tensors="pt",
    )

In [20]:
# For caching, disable generation
END_OF_SYSTEM = "<|im_end|>\n"                                                                              # Special token by Qwen, denoting the end of system block
DUMMY = render(INSTRUCTION_PROMPT=INSTRUCTION_PROMPT, CASE_PROMPT="Dummy", add_generation_prompt=False)     # Qwen requires a user turn, its content is irrelevant for caching
INSTRUCTION_CUTOFF = DUMMY.index(END_OF_SYSTEM) + len(END_OF_SYSTEM)

 # System block (INSTRUCTION_ENDS) ends right after <|im_end|>
INSTRUCTION_TEXT = DUMMY[:INSTRUCTION_CUTOFF]

# Convert INSTRUCTION_TEXT into tokens
INSTRUCTION_TOKENS = tokenizer(INSTRUCTION_TEXT, return_tensors="pt", add_special_tokens=False).to("cuda")

# Build KV cache from INSTRUCTION_TOKENS
with torch.no_grad():
    INSTRUCTION_CACHE = model(**INSTRUCTION_TOKENS, past_key_values=DynamicCache(config=model.config), use_cache=True).past_key_values

In [21]:
print(INSTRUCTION_TEXT)

<|im_start|>system
You are an expert clinician–educator.

You will receive a raw case report. Summarize the case into a JSON object for a high-quality vector database. Also make your own diagnosis from the text and justify it with reasoning and a direct quote from the case text. Follow the rules below strictly, without exceptions.

RULES (Read Carefully—No Exceptions)

1. Source Fidelity
- Use only facts from the case text.
- Do NOT invent, embellish, or “smooth out” missing data.
- Paraphrase prose into concise statements where helpful, but never add new facts.

2. Work Order
Work in this order: Metadata Skimming → Case Presentation → Diagnostic Reasoning → Final Diagnosis.

3. JSON Keys
Use exactly these top-level keys: "metadata", "case_presentation", "final_diagnosis", "diagnostic_reasoning".

4. "metadata"
- Read the case text once without summarizing it.
- "patients": one object per patient, each with "age" and "gender", as stated in the case text ("not stated" if missing). Use a

In [27]:
# For querying, supply meaningful CASE_REPORT, allow generation
CASE_TEMPLATE = "Case Report: {case_report}"
CASE_PROMPT = PromptTemplate(input_variables=["case_report"], template=CASE_TEMPLATE)
QUERY_TEXT = render(INSTRUCTION_PROMPT=INSTRUCTION_PROMPT, CASE_PROMPT=CASE_PROMPT.format(case_report=case_report), add_generation_prompt=True)

# Safety check: Cached prefix must match the start of the new prompt exactly
# print(QUERY_TEXT[:QUERY_TEXT.index(case_report)])

# For querying, supply meaningful CASE_REPORT
QUERY_TOKENS = tokenizer(QUERY_TEXT, return_tensors="pt", add_special_tokens=False).to("cuda")

QUERY_OUTPUT = model.generate(
    **QUERY_TOKENS,
    past_key_values=copy.deepcopy(INSTRUCTION_CACHE),   # Load INSTRUCTION_CACHE
    max_new_tokens=800,                                  # Limit response to 800 tokens
    do_sample=False,
)

response = tokenizer.decode(QUERY_OUTPUT[0, QUERY_TOKENS["input_ids"].shape[1]:], skip_special_tokens=True)

Store response as JSON object

In [28]:
# JSON object
print(json.dumps(response, indent=2, ensure_ascii=False))

"{\"metadata\": {\"patients\": [{\"age\": \"63\", \"gender\": \"male\"}], \"images\": [{\"image_id\": \"Image 1\", \"image_path\": \"<PMC7455599_IDR-13-2957-g0001_A_1_2.webp>\"}, {\"image_id\": \"Image 2\", \"image_path\": \"<PMC7455599_IDR-13-2957-g0001_B_2_2.webp>\"}]}, \"case_presentation\": \"A 63-year-old Thai male farmer presented with 3 months of fever, non-productive cough, left chest tightness, and 4 kg weight loss. Initially diagnosed with bacterial pneumonia, he failed to improve after 14 days of meropenem. He has a history of end-stage renal disease, kidney transplantation 4 years ago, and immunosuppression with mycophenolate mofetil, cyclosporine, and acyclovir. Physical exam showed decreased breath sounds at the left upper lung zone. Labs revealed leukocytosis (22 x 10^3/microL) and mild anemia. Imaging showed a large left upper lobe mass. Biopsy revealed non-small cell carcinoma with necrosis. Immunohistochemistry was positive for CK AE1/AE3, CK7, and TTF-1. Brown-Hopp s

#TODO:  Run the **all_cases_df** above through the LLM, obtain a JSON summary for each case:

**Phase 1.3:  Split preprocessed data into train and test set**

General function that takes care of splitting the total dataset and returning train + test set. Comes with a default algorithm which can be changed based on the function passed in.

In [37]:
def separate(df: pd.DataFrame, group_col: str = "disease", split_algorithm=None, seed: int = 42):
    if split_algorithm is None:
        split_algorithm = lambda test_proportion: 0.20        # default: always 20% test, regardless of group size

    rng = np.random.default_rng(seed)                         # default: rng seed 42
    train_parts, test_parts = [], []

    for group_val, group in df.groupby(group_col):
        n = len(group)
        test_proportion = split_algorithm(n)
        n_test = round(n * test_proportion)

        shuffled_idx = rng.permutation(group.index.to_numpy())
        test_idx = shuffled_idx[:n_test]
        train_idx = shuffled_idx[n_test:]

        test_parts.append(group.loc[test_idx])
        train_parts.append(group.loc[train_idx])

    train_df = pd.concat(train_parts, ignore_index=True)
    test_df = pd.concat(test_parts, ignore_index=True)

    return train_df, test_df

Separate dataset into train and test set, then upload both to HuggingFace repo

In [38]:
def test_fraction_for(n: int) -> float:
    """Tiered test-set fraction based on how many cases a disease has."""
    if n >= 20:
        return 0.20
    elif n >= 5:  # 5 <= n < 20
        return 0.10
    else:  # n < 5
        return 0.0

train_df, test_df = separate(df=all_cases_df, split_algorithm=test_fraction_for, seed=69420)

print(f"\nTotal: train={len(train_df)} | test={len(test_df)}")
print(train_df)


Total: train=83 | test=12
           case_id                                          case_text  \
0   PMC10870101_01  A 51-year-old female patient presented to our ...   
1    PMC6417330_01  A 24-year-old female underwent transcranial su...   
2   PMC10876100_02  A 43-year-old man, a resident of Uttar Pradesh...   
3   PMC10932498_01  A 17 year old male patient with no previous co...   
4   PMC10062082_01  A 28-year-old male presented with complaint of...   
..             ...                                                ...   
78   PMC7344764_01  A 3-year-old girl presented at our clinic with...   
79   PMC6850740_02  A 74-year-old woman, living in Tunis, Tunisia,...   
80  PMC11249797_01  A 55-year-old male presented with a medical hi...   
81   PMC8076683_01  During the 2016 ZIKV outbreak in Merida, Mexic...   
82   PMC5885377_01  A 29-year-old pregnant woman from Canada, grav...   

    disease                                             images  
0    dengue  [/home/tokuden/VGU

**Phase 1.4:  Upload preprocessed dataset to HuggingFace**

In [39]:
def verify_image_paths(df: pd.DataFrame) -> None:
    missing = []
    for case_id, paths in zip(df["case_id"], df["images"]):
        for p in paths:
            if not Path(p).is_file():
                missing.append((case_id, p))
    if missing:
        print(f"[warning] {len(missing)} missing image files, e.g.:")
        for case_id, p in missing[:5]:
            print(f"   {case_id}: {p}")
    else:
        print("All image paths verified OK.")


def build_hf_dataset(df: pd.DataFrame) -> Dataset:
    ds = Dataset.from_pandas(df.reset_index(drop=True))
    ds = ds.cast_column("images", Sequence(Image()))  # path strings -> embedded images
    return ds

In [40]:
verify_image_paths(train_df)
verify_image_paths(test_df)

REPO_ID = "choisongjun/Tropical_Diseases_Repo_1.3"
dataset_dict = DatasetDict({
    "train": build_hf_dataset(train_df.reset_index(drop=True)),
    "test": build_hf_dataset(test_df.reset_index(drop=True)),
})

print(dataset_dict["train"].features["images"])

All image paths verified OK.
All image paths verified OK.


Casting the dataset: 100%|██████████| 12/12 [00:00<00:00, 6156.02 examples/s]

List(Image(mode=None, decode=True))


In [42]:
dataset_dict.push_to_hub(REPO_ID, token=HF_TOKEN)
print(f"\nPushed to https://huggingface.co/datasets/{REPO_ID}")

Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.
Map: 100%|██████████| 83/83 [00:00<00:00, 6053.23 examples/s]

Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 127.70ba/s]
Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (0 / 1):  16%|█▌        |  713kB / 4.58MB, 58.6kB/s  
Processing Files (0 / 1):  27%|██▋       | 1.24MB / 4.58MB, 91.2kB/s  
Processing Files (0 / 1):  39%|███▊      | 1.77MB / 4.58MB,  121kB/s  
Processing Files (0 / 1):  50%|█████     | 2.29MB / 4.58MB,  143kB/s  
Processing Files (0 / 1):  62%|██████▏   | 2.82MB / 4.58MB,  173kB/s  
Processing Files (0 / 1):  73%|███████▎  | 3.34MB / 4.58MB,  191kB/s  
Processing Files (0 / 1):  85%|████████▍ | 3.87MB / 4.58MB,  207kB/s  
Processing Files (0 / 1):  96%|█████████▌| 4.40MB / 4.58MB,  233kB/s  
Processing Files (0 / 1): 100%|█████████▉| 4.58MB / 4.58MB,  235kB/s  
Processing Files (1 / 1): 100%|█████


Pushed to https://huggingface.co/datasets/choisongjun/Tropical_Diseases_Repo_1.3
